# IMDb Sentiment Analysis
Transforms text into a numerical matrix using TF-IDF Vectorization and trains a Logistic Regression model.

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

In [0]:
# Load the data into Pandas DataFrames
pandas_train_df = spark.table("imdb_sentiment_train").toPandas()
pandas_test_df = spark.table("imdb_sentiment_test").toPandas()

# Split into features (X) and target labels (y)
X_train, y_train = pandas_train_df['text'], pandas_train_df['label']
X_test, y_test = pandas_test_df['text'], pandas_test_df['label']

print(f"Training Matrix Size: {X_train.shape[0]} reviews")
print(f"Testing Matrix Size: {X_test.shape[0]} reviews")


In [0]:
# Enable MLflow auto-logging
mlflow.sklearn.autolog(log_models=False, exclusive=False)

run_name = "imdb_sklearn_run"

with mlflow.start_run(run_name=run_name) as run:
    # Capture the active run information
    active_run_id = run.info.run_id
    print(f"Active Run ID: {active_run_id}")

    # 1. Instantiate with default or base settings
    pipeline = Pipeline([
        ('vectorizer', TfidfVectorizer()),
        ('classifier', LogisticRegression())
    ])

    # 2. Update hyperparameters dynamically before fitting
    max_features = 5000
    stop_words = 'english'
    regularization_C = 1.0
    max_iter = 1000
    random_state = 42
    
    pipeline.set_params(
        vectorizer__max_features=max_features,
        vectorizer__stop_words=stop_words,
        classifier__C=regularization_C,
        classifier__max_iter=max_iter,
        classifier__random_state=random_state
    )
    
    # This automatically runs fit_transform on the vectorizer, 
    # then passes the resulting matrix straight into the Logistic Regression
    pipeline.fit(X_train, y_train)
    
    # Calculate accuracy
    accuracy = pipeline.score(X_test, y_test)
    mlflow.log_metric("test_accuracy", accuracy)
    
    print(f"Test Accuracy: {accuracy:.4f}")

    # 1. Format your input example as a 1D NumPy array of strings
    # Do NOT use a dictionary or a Pandas DataFrame here
    text_input_example = np.array(["sample review text one", "sample review text two"], dtype=str)

    # 2. Compute predictions on that 1D array to get the output format
    dummy_predictions = pipeline.predict(text_input_example)

    # 3. Explicitly infer the signature
    # This forces MLflow to register a TensorSpec [-1] or flat text layout 
    # instead of a named column layout
    signature = infer_signature(text_input_example, dummy_predictions)

    # 4. Log the model using mlflow.sklearn
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        signature=signature,              # <-- Forces the flat array signature
        input_example=text_input_example  # <-- Shows it accepts 1D arrays
    )

In [0]:
# 1. Register the scikit-learn model
mlflow.register_model(
    model_uri=f"runs:/{active_run_id}/model",
    name="imdb_sklearn_model"
)